<div align="center">

<h1>Track 7: Stranded Assets</h1>

<h3>How much fossil capacity would have to retire before the end of its technical life along the different transition pathways, and where is that concentrated, by country, technology and plant age?</h3>

</div>

---

Course: Environmental Economics, BSc AIDAMS

Teacher: Caterina Seghini, ESSEC Department of Economics

Date: October 2026

### Group members

- Edgar Gusching
- Adrien Labergere
- Louis Bettelli
- Oscar Moulet
- Lucas Magret Delord
- Victor Henz

## About this notebook

This notebook produces all the tables and figures used in the report, starting from the raw files as downloaded. The notes between the cells explain what each step does and which choices are made in the code. The discussion of the results is in the report.

### Input files

Put these three files, unchanged, in a folder called data next to this notebook.

| Producer and dataset | File | Version | Accessed |
|---|---|---|---|
| Global Energy Monitor, Global Coal Plant Tracker | Global-Coal-Plant-Tracker-July-2026.xlsx | July 2026 | 14 Oct 2026 |
| Global Energy Monitor, Global Oil and Gas Plant Tracker | Global Oil and Gas Plant Tracker (GOGPT) - August 2026.xlsx | August 2026 | 14 Oct 2026 |
| NGFS and IIASA, NGFS Climate Scenarios Data Set | 1773763507243-V5.1_NGFS_Phase_5.zip (we use IAM_data.xlsx inside it) | v5.1, 20 March 2026, DOI 10.5281/zenodo.17901363 | 14 Oct 2026 |

The two Global Energy Monitor files come from a form on the Global Energy Monitor download page (licence CC BY 4.0). The NGFS zip comes from the NGFS Scenario Explorer (data.ece.iiasa.ac.at/ngfs), with the Guest login, in the Downloads section.

Run all cells from top to bottom. Each run reads the original files, including the NGFS ZIP, which takes a few minutes. Only the figures and LaTeX tables used by the report are saved; the other results are displayed here.

In [ ]:
# Import the libraries we need
from pathlib import Path
import zipfile
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import country_converter as coco

# Where the raw files are, and where the figures and tables are saved
# These relative paths work as long as Jupyter is opened from the project folder.
DATA_FOLDER = Path("data")
FIGURE_FOLDER = Path("figures")
FIGURE_FOLDER.mkdir(exist_ok=True)

COAL_FILE = DATA_FOLDER / "Global-Coal-Plant-Tracker-July-2026.xlsx"
GAS_FILE = DATA_FOLDER / "Global Oil and Gas Plant Tracker (GOGPT) - August 2026.xlsx"
NGFS_ZIP = DATA_FOLDER / "1773763507243-V5.1_NGFS_Phase_5.zip"
NGFS_FILE_IN_ZIP = "V5.1 NGFS Phase 5/IAM_data.xlsx"

# Let the wider summary tables display without hiding most of their columns.
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

: 

In [ ]:
# The year we treat as "today", to compute the age of each plant
BASE_YEAR = 2026

# The three models and the three scenarios we use
# Keeping the model list fixed gives each scenario the same three-model comparison.
MODELS = [
    "GCAM 6.0 NGFS",
    "MESSAGEix-GLOBIOM 2.0-M-R12-NGFS",
    "REMIND-MAgPIE 3.3-4.8",
]
SCENARIOS = ["Net Zero 2050", "Delayed transition", "Current Policies"]

# The five world regions (R5) reported by all three models
R5_REGIONS = [
    "Asia (R5)",
    "Latin America (R5)",
    "Middle East & Africa (R5)",
    "OECD & EU (R5)",
    "Reforming Economies (R5)",
]

In [ ]:
# Chart settings used for every figure (one colour per scenario)
SCENARIO_COLORS = {
    "Net Zero 2050": "#2a78d6",
    "Delayed transition": "#eb6834",
    "Current Policies": "#1baf7a",
}
COAL_COLOR = "#2a78d6"
GAS_COLOR = "#eb6834"
INK = "#0b0b0b"
GREY = "#52514e"
LIGHT_GREY = "#d8d7d2"

# Set the appearance once so the saved figures use the same fonts and spacing.
plt.rcParams.update({
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": LIGHT_GREY,
    "grid.linewidth": 0.6,
    "legend.frameon": False,
})

## 1. Loading the plant data

Each row of the two trackers is one generating unit. We keep the same columns for coal and gas so that we can stack them in one table.

Choices made here:

- Gas: the oil and gas tracker also contains oil plants. We keep units whose fuel classification is Gas, Multi Fuel or LNG only, and drop Oil. Multi-fuel units can burn gas, so they count as gas capacity.
- Technology groups: coal is one group. Gas is split into combined cycle, gas turbine, steam turbine, and "other/unknown" for everything else, because these types last for very different lengths of time (section 5).
- We do not use the coal column "Remaining plant lifetime (years)". It is a derived assumption, not an observed technical lifetime.
- Only the sheets "Units" (coal) and "Gas & Oil Units" (gas) are used.

In [ ]:
# Read the coal tracker
coal_raw = pd.read_excel(COAL_FILE, sheet_name="Units")
print("Coal tracker:", len(coal_raw), "rows")

# Keep the columns we need, with short names
# Use the same column names for both trackers so we can analyse them together.
# Unreadable numbers become NaN; we check missing values before using them.
coal = pd.DataFrame()
coal["unit_id"] = coal_raw["GEM unit/phase ID"]
coal["country"] = coal_raw["Country/Area"].astype(str).str.strip()
coal["capacity_mw"] = pd.to_numeric(coal_raw["Capacity (MW)"], errors="coerce")
# Standardise spaces and case so later status filters match consistently.
coal["status"] = coal_raw["Status"].astype(str).str.strip().str.lower()
coal["start_year"] = pd.to_numeric(coal_raw["Start year"], errors="coerce")
coal["retired_year"] = pd.to_numeric(coal_raw["Retired year"], errors="coerce")
coal["technology"] = coal_raw["Combustion technology"].astype(str).str.strip()
coal["fuel"] = "Coal"
coal["tech_group"] = "Coal (all)"
coal.head(3)

In [ ]:
# Read the oil and gas tracker
gas_raw = pd.read_excel(GAS_FILE, sheet_name="Gas & Oil Units")
print("Oil and gas tracker:", len(gas_raw), "rows")
print(gas_raw["Fuel classification"].value_counts())

# Drop the units that burn oil only
# Multi-fuel units stay in the sample because they can burn gas.
# This choice affects the gas total, so we quantify its size in the next few cells.
gas_raw = gas_raw[gas_raw["Fuel classification"].isin(["Gas", "Multi Fuel", "LNG only"])]

# Give the gas data the same structure and units as the coal data.
gas = pd.DataFrame()
gas["unit_id"] = gas_raw["GEM unit ID"]
gas["country"] = gas_raw["Country/Area"].astype(str).str.strip()
gas["capacity_mw"] = pd.to_numeric(gas_raw["Capacity (MW)"], errors="coerce")
gas["status"] = gas_raw["Status"].astype(str).str.strip().str.lower()
gas["start_year"] = pd.to_numeric(gas_raw["Start year"], errors="coerce")
gas["retired_year"] = pd.to_numeric(gas_raw["Retired year"], errors="coerce")
gas["technology"] = gas_raw["Turbine/Engine Technology"].astype(str).str.strip()
gas["fuel"] = "Gas"

In [ ]:
# Put each gas unit in one of four technology groups
main_types = ["combined cycle", "gas turbine", "steam turbine"]

tech_groups = []
for tech in gas["technology"]:
    if tech in main_types:
        tech_groups.append("Gas: " + tech)
    else:
        # Keep uncommon or missing technology labels rather than dropping those units.
        tech_groups.append("Gas: other/unknown")
gas["tech_group"] = tech_groups

gas["tech_group"].value_counts()

In [ ]:
# Stack coal and gas into one table
# This stacks rows; it does not match coal units to gas units.
# We keep all statuses here because retired units are needed for the lifetime description.
plants = pd.concat([coal, gas], ignore_index=True)
print(len(plants), "units in", plants["country"].nunique(), "countries or areas")

# Number of units by status and fuel
pd.crosstab(plants["status"], plants["fuel"])

In [ ]:
# How much operating gas capacity is multi-fuel (to show what dropping them would change)
operating_gas = gas_raw[gas_raw["Status"].str.lower() == "operating"]
gas_mw = pd.to_numeric(operating_gas["Capacity (MW)"], errors="coerce")
# Weight by MW rather than by unit count: a large unit contributes more to the fleet.
multi_mw = gas_mw[operating_gas["Fuel classification"] == "Multi Fuel"].sum()
print(f"Multi-fuel share of operating gas capacity: {multi_mw / gas_mw.sum() * 100:.0f}%")

## 2. Giving each plant a region

The NGFS models do not give results by country, and each model has its own regions. The only regions that all three models share are the five R5 regions, so we compare plants and scenarios at that level.

How we do it: the country_converter package gives each country its MESSAGE region (11 regions), and we group these into the five R5 regions with the table below. A few small territories are not recognised by the package, so we assign them by hand.

In [ ]:
# How the 11 MESSAGE regions are grouped into the 5 R5 regions
# These codes come from country_converter, not from the plant trackers.
MESSAGE_TO_R5 = {
    "CPA": "Asia (R5)",
    "PAS": "Asia (R5)",
    "SAS": "Asia (R5)",
    "LAC": "Latin America (R5)",
    "MEA": "Middle East & Africa (R5)",
    "AFR": "Middle East & Africa (R5)",
    "EEU": "OECD & EU (R5)",
    "NAM": "OECD & EU (R5)",
    "PAO": "OECD & EU (R5)",
    "WEU": "OECD & EU (R5)",
    "FSU": "Reforming Economies (R5)",
}

# Places that country_converter does not recognise, assigned by hand
# Include alternative spellings because the source files do not always use the same names.
MANUAL_R5 = {
    "Kosovo": "OECD & EU (R5)",
    "Sint Maarten": "Latin America (R5)",
    "Curacao": "Latin America (R5)",
    "Curaçao": "Latin America (R5)",
    "Bonaire": "Latin America (R5)",
    "Saba": "Latin America (R5)",
    "Sint Eustatius": "Latin America (R5)",
    "Netherlands Antilles": "Latin America (R5)",
    "Western Sahara": "Middle East & Africa (R5)",
    "Macau": "Asia (R5)",
    "Macao": "Asia (R5)",
    "Hong Kong": "Asia (R5)",
}

In [ ]:
# Find the MESSAGE region of every country, then its R5 region
converter = coco.CountryConverter()
# Convert each distinct country once, then reuse the lookup for every unit.
country_list = sorted(plants["country"].unique())
message_codes = converter.convert(country_list, to="MESSAGE", not_found=None)

country_to_r5 = {}
for country, message_code in zip(country_list, message_codes):
    if message_code in MESSAGE_TO_R5:
        country_to_r5[country] = MESSAGE_TO_R5[message_code]
    else:
        # Use our territory lookup only when the automatic mapping has no usable region.
        country_to_r5[country] = MANUAL_R5.get(country)

plants["r5"] = plants["country"].map(country_to_r5)

# Every unit should have a region
missing = plants[plants["r5"].isna()]["country"].unique()
print("Countries without a region:", list(missing))
# Stop here if a country is missing, otherwise later regional sums would silently omit it.
assert len(missing) == 0, "Unmapped plant countries"

## 3. Loading the NGFS scenarios

We read IAM_data.xlsx directly from the zip and keep four variables for the three models:

- Capacity|Electricity|Coal and Capacity|Electricity|Gas, in GW. These are the ones we compare with the plants.
- Secondary Energy|Electricity|Coal and Secondary Energy|Electricity|Gas, in EJ per year (electricity generated). Only used for the check in section 11.

The brief talks about the path of generation. We use capacity because it is in the same unit as the plant data (GW). Going from generation to capacity would need a guess of how many hours per year plants run, in every region and every year.

The file has one column per year (every 5 years). We reshape it to one row per model, scenario, region, variable and year.

In [ ]:
# Keep installed capacity for the main analysis and generation for the utilisation check.
NGFS_VARIABLES = [
    "Capacity|Electricity|Coal",
    "Capacity|Electricity|Gas",
    "Secondary Energy|Electricity|Coal",
    "Secondary Energy|Electricity|Gas",
]

# Read the original download on every run for reproducibility.
print("Reading IAM_data.xlsx from the zip, this takes a few minutes...")
with zipfile.ZipFile(NGFS_ZIP) as z:
    with z.open(NGFS_FILE_IN_ZIP) as f:
        ngfs_raw = pd.read_excel(f, sheet_name="data")
# Filter before reshaping: the download contains many variables and models we do not use.
ngfs_raw = ngfs_raw[ngfs_raw["Variable"].isin(NGFS_VARIABLES) & ngfs_raw["Model"].isin(MODELS)]

# One column per year -> one row per year
year_columns = [c for c in ngfs_raw.columns if str(c).strip().isdigit()]
# Each observation will now identify one model, scenario, region, variable and year.
ngfs = ngfs_raw.melt(id_vars=["Model", "Scenario", "Region", "Variable", "Unit"],
                     value_vars=year_columns, var_name="year", value_name="value")
ngfs = ngfs.rename(columns={"Model": "model", "Scenario": "scenario",
                            "Variable": "variable", "Unit": "unit"})
ngfs["year"] = ngfs["year"].astype(int)
# Some region names start with the model name and a "|", we keep the last part
ngfs["region"] = ngfs["Region"].str.split("|").str[-1]
# Both variable names end in Coal or Gas, which gives us a common fuel label.
ngfs["fuel"] = ngfs["variable"].str.split("|").str[-1]
ngfs = ngfs[["model", "scenario", "region", "variable", "fuel", "unit", "year", "value"]]

print(len(ngfs), "rows")
print(ngfs.groupby(["variable", "unit"]).size())


In [ ]:
# A separate table with only the capacity variables (GW)
capacity = ngfs[ngfs["variable"].str.startswith("Capacity")].copy()
capacity = capacity.rename(columns={"value": "capacity_gw"})
capacity.head(3)
# Interpolate 2026 capacity separately for each model, scenario, region and fuel.
keys = ["model", "scenario", "region", "variable", "fuel", "unit"]
# Matching indices make sure we interpolate between the same series in both years.
a = capacity[capacity.year == 2025].set_index(keys).sort_index()
b = capacity[capacity.year == 2030].set_index(keys).sort_index()
assert a.index.equals(b.index)
base = a.copy()
# 2026 is one fifth of the way from 2025 to 2030; assume a straight line between them.
base["capacity_gw"] = a["capacity_gw"] + (b["capacity_gw"] - a["capacity_gw"]) / 5
base["year"] = BASE_YEAR
capacity = pd.concat([capacity, base.reset_index()], ignore_index=True)
# Check the exact coverage needed below, before averaging anything across models.
selected = capacity[capacity.scenario.isin(SCENARIOS) & capacity.region.isin(R5_REGIONS + ["World"])
                    & capacity.year.isin([2026, 2030, 2035, 2040, 2045, 2050])]
assert selected["capacity_gw"].notna().all()
assert not selected.duplicated(["model", "scenario", "region", "fuel", "year"]).any()
# Expected rows: 3 models x 3 scenarios x 6 regions x 2 fuels x 6 dates.
assert len(selected) == 3 * 3 * 6 * 2 * 6

Two checks on geography: selected native MESSAGE regions reproduce their R5 aggregates, and R5 totals reproduce World for every model, scenario, fuel and comparison year used below. Execution stops if the tolerance is exceeded.

In [ ]:
# Check 1: MESSAGE native regions added up inside each R5 region = R5 value reported by MESSAGE
MESSAGE_R12_TO_R5 = {
    "China": "Asia (R5)",
    "Rest Centrally Planned Asia": "Asia (R5)",
    "Other Pacific Asia": "Asia (R5)",
    "South Asia": "Asia (R5)",
    "Latin America and the Caribbean": "Latin America (R5)",
    "Middle East and North Africa": "Middle East & Africa (R5)",
    "Sub-saharan Africa": "Middle East & Africa (R5)",
    "Eastern Europe": "OECD & EU (R5)",
    "North America": "OECD & EU (R5)",
    "Pacific OECD": "OECD & EU (R5)",
    "Western Europe": "OECD & EU (R5)",
    "Former Soviet Union": "Reforming Economies (R5)",
}

# Use one scenario and a few dates to check our MESSAGE regional grouping.
message = capacity[(capacity["model"] == "MESSAGEix-GLOBIOM 2.0-M-R12-NGFS")
                   & (capacity["scenario"] == "Current Policies")
                   & (capacity["year"].isin([2020, 2030, 2040, 2050]))]

for fuel in ["Coal", "Gas"]:
    this_fuel = message[message["fuel"] == fuel]
    for r5 in R5_REGIONS:
        members = [name for name, region in MESSAGE_R12_TO_R5.items() if region == r5]
        reported = this_fuel[this_fuel["region"] == r5].groupby("year")["capacity_gw"].sum()
        added_up = this_fuel[this_fuel["region"].isin(members)].groupby("year")["capacity_gw"].sum()
        # Allow a small rounding difference, but stop if the regional totals disagree.
        difference = (reported - added_up).abs().max()
        print(f"{fuel:<5} {r5:<27} largest difference: {difference:.3f} GW")
        assert difference < 0.001

In [ ]:
# Check R5 aggregation for all selected models, scenarios, fuels and years.
keys = ["model", "scenario", "fuel", "year"]
reported = selected[selected.region == "World"].set_index(keys)["capacity_gw"].sort_index()
regional_sum = selected[selected.region.isin(R5_REGIONS)].groupby(keys)["capacity_gw"].sum().sort_index()
# Check that the series cover the same combinations before comparing their values.
assert reported.index.equals(regional_sum.index)
# 0.001 GW is 1 MW. Use an absolute tolerance so large regions do not get extra leeway.
assert np.allclose(reported, regional_sum, atol=0.001, rtol=0)
print("All selected R5 sums match World within 0.001 GW.")

## 4. Describing the operating fleet

From here on, "the fleet" means units with status "operating". Units under construction are not included: they are not yet operating.

Table 1 is displayed below. Age is BASE_YEAR minus the start year.

In [ ]:
# Only the units running today
operating = plants[plants["status"] == "operating"].copy()
operating["age"] = BASE_YEAR - operating["start_year"]

# Table 1: number of units, capacity, unit size and age, by technology group
# Counts and capacity include units with missing start years.
# pandas leaves those missing ages out of the age statistics.
# The mean age is a mean across units, not a capacity-weighted mean.
table1 = operating.groupby(["fuel", "tech_group"]).agg(
    units=("unit_id", "size"),
    capacity_gw=("capacity_mw", "sum"),
    mean_unit_mw=("capacity_mw", "mean"),
    sd_unit_mw=("capacity_mw", "std"),
    mean_age=("age", "mean"),
    sd_age=("age", "std"),
    oldest_start=("start_year", "min"),
    newest_start=("start_year", "max"),
    countries=("country", "nunique"),
)
# The trackers report MW; divide by 1,000 whenever we present capacity in GW.
table1["capacity_gw"] = table1["capacity_gw"] / 1000

# Add a row for the whole fleet
# Calculate these totals from the units themselves, not from the group averages.
table1.loc[("Total", "all"), :] = [
    len(operating),
    operating["capacity_mw"].sum() / 1000,
    operating["capacity_mw"].mean(),
    operating["capacity_mw"].std(),
    operating["age"].mean(),
    operating["age"].std(),
    operating["start_year"].min(),
    operating["start_year"].max(),
    operating["country"].nunique(),
]
table1 = table1.round(1)
table1 = table1.astype({"units": int, "oldest_start": int, "newest_start": int, "countries": int})
table1

In [ ]:
# Missing data: operating units with no start year cannot be given an age, so they are dropped later
no_start = operating[operating["start_year"].isna()]
print("Operating units with no start year:", len(no_start),
      "| capacity:", round(no_start["capacity_mw"].sum() / 1000, 1), "GW")
print(no_start.groupby("fuel").agg(units=("unit_id", "size"), missing_start_mw=("capacity_mw", "sum")))
print("Missing operating capacity:", operating["capacity_mw"].isna().sum())
# These checks catch inputs that could distort the capacity sums or the oldest-first ordering.
# A failed assertion stops the run so the issue cannot pass unnoticed into the results.
assert operating["capacity_mw"].notna().all()
assert (operating["capacity_mw"] > 0).all()
assert not operating.duplicated(["fuel", "unit_id"]).any()
assert operating["unit_id"].notna().all()
assert (operating["start_year"].dropna() <= BASE_YEAR).all()
# Report construction separately; these units never enter our operating-fleet path.
construction_coal_gw = plants.loc[(plants.fuel == "Coal") & (plants.status == "construction"), "capacity_mw"].sum() / 1000
print(f"Coal capacity under construction (excluded): {construction_coal_gw:.1f} GW")

In [ ]:
# Operating capacity (GW) by region and fuel
# The rows are regions and the columns are fuels, making the geographic split easy to compare.
by_region = operating.pivot_table(index="r5", columns="fuel", values="capacity_mw", aggfunc="sum") / 1000
by_region["Total"] = by_region["Coal"] + by_region["Gas"]
by_region.sort_values("Total", ascending=False).round(0)

In [ ]:
# The 15 countries with the most operating coal and gas capacity (GW)
# Fill an absent country-fuel combination with zero before adding the two fuels.
by_country = operating.pivot_table(index="country", columns="fuel", values="capacity_mw",
                                   aggfunc="sum", fill_value=0) / 1000
by_country["Total"] = by_country["Coal"] + by_country["Gas"]
by_country.sort_values("Total", ascending=False).head(15).round(0)

In [ ]:
# When the operating fleet was built: capacity (GW) by decade of start year
# Floor the start year to its decade, so 2007 becomes 2000.
# Nullable Int64 keeps units with unknown start years marked as missing.
operating["decade_built"] = (operating["start_year"] // 10 * 10).astype("Int64")
operating.pivot_table(index="decade_built", columns="fuel", values="capacity_mw",
                      aggfunc="sum", fill_value=0).div(1000).round(0)

In [ ]:
# Same thing for the 5 biggest countries: capacity (GW) built in each decade, coal and gas together
# Choose countries by their total operating capacity, then compare their construction histories.
# This display starts in 1960; earlier units still count in the fleet totals above.
top_5 = by_country.sort_values("Total", ascending=False).head(5).index
recent = operating[operating["country"].isin(top_5) & (operating["decade_built"] >= 1960)]
recent.pivot_table(index="country", columns="decade_built", values="capacity_mw",
                   aggfunc="sum", fill_value=0).div(1000).round(0)

## 5. Observed retirement ages and assumed lifetimes

For retired units with both dates, observed retirement age is retirement year minus start year. Values outside 0-100 years are excluded from this description. We report medians, quartiles and sample sizes, not a fitted survival distribution.

The records cover closures from 2000 for coal and 2020 for gas. Earlier closures are absent, and operating units have not completed their lives. These selected retirement ages do not identify the technical lifetime of today's fleet. We use them as context for explicit assumptions and test those assumptions over a wide range.

In [ ]:
# Closed units with a start year and a retirement year
retired = plants[(plants["status"] == "retired")
                 & plants["start_year"].notna()
                 & plants["retired_year"].notna()].copy()
# This is the age at a recorded retirement, not an estimate of a physical expiry date.
retired["lifetime"] = retired["retired_year"] - retired["start_year"]
# Exclude negative ages and ages above 100 from this descriptive sample.
retired = retired[(retired["lifetime"] >= 0) & (retired["lifetime"] <= 100)]

print(len(retired), "closed units with a usable lifetime")
print("Retirement years go from", int(retired["retired_year"].min()), "to", int(retired["retired_year"].max()))
# Show the observation windows as well as the sample sizes; they differ sharply by fuel.
print(retired.groupby("fuel")["retired_year"].agg(["min", "max", "count"]))

In [ ]:
# Observed retirement ages: descriptive statistics for recorded closures only
lifetimes_by_group = retired.groupby("tech_group")["lifetime"]

table2 = pd.DataFrame()
table2["closed_units"] = lifetimes_by_group.count()
table2["mean"] = lifetimes_by_group.mean()
table2["median"] = lifetimes_by_group.median()
# The middle half of recorded retirement ages lies between these two quartiles.
# Small groups should be read with care, even when their median looks precise.
table2["p25"] = lifetimes_by_group.quantile(0.25)
table2["p75"] = lifetimes_by_group.quantile(0.75)


table2.round(1)

### Lifetime assumptions

- Coal: 45 years, a rounded scenario assumption close to the historical global mean of about 46 years discussed by Cui et al. (2019), who also examine 50-year operation. Our recorded-closure median is 39 years; it is not the lifetime of all plants.
- Gas: 30 years for combined cycle, 35 for gas turbines and other/unknown, and 45 for steam turbines. These are illustrative values informed by observed retirement ages, not estimated technical limits. The small and selected gas sample does not validate the levels or ranking for today's fleet.

Section 10 varies all assumptions by -10 to +10 years. We do not fit a survival estimator or report confidence intervals. Source: https://doi.org/10.1038/s41467-019-12618-3

In [ ]:
# The lifetime we assume for each technology group (years)
# These are scenario assumptions. We do not set them equal to the observed medians.
# Later we shift all of them by up to ten years to see how much the results move.
LIFETIMES = {
    "Coal (all)": 45,
    "Gas: combined cycle": 30,
    "Gas: gas turbine": 35,
    "Gas: steam turbine": 45,
    "Gas: other/unknown": 35,
}

# Align by technology name to put the assumption beside the observed retirement ages.
table2["assumed"] = pd.Series(LIFETIMES)
table2.round(1)

In [ ]:
# Figure 1: observed retirement ages, not estimated technical lives.
groups = list(LIFETIMES)
labels = [g.replace("Gas: ", "Gas, ") for g in groups]
fig, ax = plt.subplots(figsize=(7.4, 2.8))
# Each box summarises the recorded closures in one technology group.
# The points outside the whiskers are still observations, not automatically data errors.
ax.boxplot([retired.loc[retired.tech_group == g, "lifetime"] for g in groups],
           orientation="horizontal", widths=0.55, flierprops=dict(marker=".", markersize=2))
ax.set_yticks(range(1, len(groups)+1), labels)
# Add sample sizes so the reader can see which boxes rest on very few closures.
for i,g in enumerate(groups, start=1):
    ax.text(100,i+0.25,f"n = {(retired.tech_group == g).sum():,}",ha="right",fontsize=8)
ax.set_xlabel("Observed retirement age (years)")
ax.set_xlim(0,102)
ax.set_title("Recorded closures: coal 2000-2026; gas 2020-2026",loc="left")
fig.savefig(FIGURE_FOLDER / "fig1_lifetimes.png")
plt.show()

Some operating units already exceed their group's assumed lifetime. They are excluded from the age-limited committed path immediately. The table quantifies this limitation: the assumptions are stylised thresholds, not physical expiry dates.

In [ ]:
# Operating capacity already older than its assumed lifetime
check = operating[operating["start_year"].notna() & operating["capacity_mw"].notna()].copy()
check["lifetime"] = check["tech_group"].map(LIFETIMES)
# Equality counts here: our committed path excludes a unit in its natural retirement year.
check["past_life"] = check["age"] >= check["lifetime"]

# Compare excluded capacity with the known-age operating stock in the same group.
# A large share is a reminder that the assumed lifetime is only a modelling threshold.
past_life = pd.DataFrame()
past_life["operating_gw"] = check.groupby("tech_group")["capacity_mw"].sum() / 1000
past_life["past_life_gw"] = check[check["past_life"]].groupby("tech_group")["capacity_mw"].sum() / 1000
past_life["share_%"] = past_life["past_life_gw"] / past_life["operating_gw"] * 100
past_life.round(1)

## 6. Committed capacity

For the 2026 operating inventory, natural retirement year = start year + assumed lifetime. A unit contributes only if its start year is at or before the evaluation year and its natural retirement is after it. The path starts in 2026, adds no new units and excludes units already beyond the assumed lifetime.

In [ ]:
# Build a no-new-build capacity path for a set of units and lifetime assumptions.
# Return one row per requested year, with capacity in GW.
def committed_capacity(units, lifetimes, years):
    # Capacity (GW) of today's operating units still within their lifetime in each year
    running = units[(units["status"] == "operating")
                    & units["start_year"].notna()
                    & units["capacity_mw"].notna()].copy()
    # Match each technology to its assumed lifetime and add it to the commissioning year.
    running["natural_retirement"] = running["start_year"] + running["tech_group"].map(lifetimes)

    rows = []
    for year in years:
        # A unit contributes only before its assumed retirement year.
        # Units already beyond the threshold at the start are excluded immediately.
        still_running = running[(running["start_year"] <= year) & (running["natural_retirement"] > year)]
        # Sum MW first, then convert the total to GW. No early closures are applied here.
        rows.append({"year": year, "committed_gw": still_running["capacity_mw"].sum() / 1000})
    return pd.DataFrame(rows)

In [ ]:
# Evaluate the 2026 inventory at the start date and the five-year scenario dates.
YEARS = [2026, 2030, 2035, 2040, 2045, 2050]

# Run the same calculation separately for coal and gas, then combine the two paths.
committed_list = []
for fuel in ["Coal", "Gas"]:
    path = committed_capacity(plants[plants["fuel"] == fuel], LIFETIMES, YEARS)
    path["fuel"] = fuel
    committed_list.append(path)
committed = pd.concat(committed_list, ignore_index=True)

committed.pivot_table(index="year", columns="fuel", values="committed_gw").round(0)

## 7. The scenario paths

For each fuel, scenario and year we take the World value of each model and average the three models. The second table shows each model separately in 2040, because the models disagree a lot and the average hides that.

In [ ]:
# World capacity for each fuel, scenario and year, averaged over the three models
world = capacity[(capacity["region"] == "World") & capacity["scenario"].isin(SCENARIOS)]
# Each of the three models gets equal weight. This is a scenario summary, not a forecast.
scen = world.groupby(["fuel", "scenario", "year"], as_index=False)["capacity_gw"].mean()
scen = scen[(scen["year"] >= 2025) & (scen["year"] <= 2050)]

scen.pivot_table(index=["fuel", "scenario"], columns="year", values="capacity_gw").round(0)

In [ ]:
# Each model separately, in 2040 (GW)
# Keeping the models visible shows the spread that the average in the previous cell hides.
world[world["year"] == 2040].pivot_table(index=["fuel", "scenario"], columns="model",
                                         values="capacity_gw").round(0)

The 2026 inventory and NGFS are different measurements. We compare the full observed operating stock and the subset within assumed life with each scenario's interpolated 2026 capacity. We do not force their starting levels to match or interpret the difference as a measured policy effect.

In [ ]:
# Compare three starting stocks: the observed fleet, the part within assumed life,
# and the scenario level. Differences in coverage can already produce a gap in 2026.
base_rows = []
for fuel in ["Coal", "Gas"]:
    observed = operating.loc[operating.fuel == fuel, "capacity_mw"].sum() / 1000
    within_life = committed.loc[(committed.fuel == fuel) & (committed.year == BASE_YEAR), "committed_gw"].iloc[0]
    for scenario in SCENARIOS:
        scenario_base = scen.loc[(scen.fuel == fuel) & (scen.scenario == scenario) & (scen.year == BASE_YEAR), "capacity_gw"].iloc[0]
        base_rows.append(dict(fuel=fuel, scenario=scenario, observed_gw=observed,
                              within_life_gw=within_life, scenario_gw=scenario_base))
# Keep these differences visible rather than forcing all paths to share a starting value.
baseline = pd.DataFrame(base_rows)
baseline.round(1)

## 8. Excess capacity relative to a scenario

For each fuel and date, gap = max(committed capacity - scenario capacity, 0). This is a conditional excess-capacity stock at that date, not annual or cumulative retirements and not a forecast. The scenario may contain new plants and retrofits not modelled in our fixed inventory.

In [ ]:
# Figure 2: committed path against the three scenarios, coal and gas
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2))
fig.subplots_adjust(wspace=0.75)

# One panel per fuel; the dashed path uses only today's operating units.
for ax, fuel in zip(axes, ["Coal", "Gas"]):
    c = committed[committed["fuel"] == fuel]
    ax.plot(c["year"], c["committed_gw"], color=INK, linewidth=2, linestyle="--")
    ax.annotate("Committed\n(no early closure)", xy=(2050, c["committed_gw"].iloc[-1]),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=7.5)

    for scenario, color in SCENARIO_COLORS.items():
        s = scen[(scen["fuel"] == fuel) & (scen["scenario"] == scenario) & scen["year"].isin(YEARS)]
        ax.plot(s["year"], s["capacity_gw"], color=color, linewidth=2)
        ax.annotate(scenario, xy=(2050, s["capacity_gw"].iloc[-1]), xytext=(4, 0),
                    textcoords="offset points", va="center", fontsize=7.5, color=color)
        # Shade the gap between the committed path and the scenario
        # Match by year before shading, and shade only where committed capacity is higher.
        both = s.merge(c, on="year")
        ax.fill_between(both["year"], both["capacity_gw"], both["committed_gw"],
                        where=both["committed_gw"] > both["capacity_gw"],
                        color=color, alpha=0.07, linewidth=0)

    ax.set_title(f"{fuel}-fired capacity, world", loc="left")
    ax.set_xlabel("Year")
    ax.set_ylabel("Capacity (GW)")
    ax.set_xlim(2026, 2066)  # space on the right for the labels
    ax.set_xticks([2030, 2040, 2050])
    ax.set_ylim(bottom=0)

fig.savefig(FIGURE_FOLDER / "fig2_committed_vs_scenarios.png")
plt.show()

In [ ]:
# The gap (GW) for each fuel, scenario and year
rows = []
for fuel in ["Coal", "Gas"]:
    c = committed[committed["fuel"] == fuel]
    for scenario in SCENARIOS:
        s = scen[(scen["fuel"] == fuel) & (scen["scenario"] == scenario)]
        # Join by evaluation year so the two capacities refer to the same date.
        both = c.merge(s, on="year")
        for i, row in both.iterrows():
            # If the scenario allows more capacity than we retain, the excess is zero.
            # This is a stock at that date, not cumulative closures over the whole period.
            gap = max(row["committed_gw"] - row["capacity_gw"], 0)
            rows.append({"fuel": fuel, "scenario": scenario, "year": int(row["year"]), "gap_gw": gap})
gaps = pd.DataFrame(rows)

gaps.pivot_table(index=["fuel", "scenario"], columns="year", values="gap_gw").round(0)

## 9. Allocating regional excess capacity

For Net Zero 2050 in 2040, compute each fuel's gap within each R5 region. Allocate it to the oldest units still within assumed life. Break equal-start-year ties by unit ID. The last unit can receive a fractional allocation so the total matches the gap exactly; this is accounting, not a proposal to retire part of a physical unit.

Country and technology results are conditional allocations. Ages are ages in 2040, not estimated closure dates. Regional gaps can sum to more than the world gap if a regional surplus cannot offset a shortage elsewhere.

In [ ]:
# Allocate one regional fuel gap to eligible units, oldest first.
# Return all eligible units with their allocated MW, including zeros for units not needed.
def stranded_units(units, lifetimes, target_year, scenario_gw):
    # Start with the same known-age operating sample used for committed capacity.
    running = units[(units.status == "operating") & units.start_year.notna()
                    & units.capacity_mw.notna()].copy()
    running["natural_retirement"] = running.start_year + running.tech_group.map(lifetimes)
    running = running[(running.start_year <= target_year) & (running.natural_retirement > target_year)]
    # Earlier commissioning means older. Unit ID makes equal-age ties reproducible.
    running = running.sort_values(["start_year", "unit_id"], kind="stable").copy()
    # Convert the scenario target to MW to match the individual units.
    gap_mw = max(running.capacity_mw.sum() - scenario_gw * 1000, 0)
    # For each unit, work out how much capacity the older units have already supplied.
    cumulative_before = running.capacity_mw.cumsum() - running.capacity_mw
    # Give this unit only the gap still left, capped at its own capacity.
    # The boundary unit can receive a fraction; later units receive zero.
    running["allocated_mw"] = (gap_mw - cumulative_before).clip(lower=0).clip(upper=running.capacity_mw)
    # Ignore tiny floating-point residues when flagging units with a positive allocation.
    running["allocated"] = running.allocated_mw > 1e-8
    # These ages and remaining years are measured at the target date, not an assigned closure date.
    running["age_in_target_year"] = target_year - running.start_year
    running["remaining_life_at_target"] = running.natural_retirement - target_year
    # Check that we filled the gap exactly and never allocated more MW than a unit contains.
    assert np.isclose(running.allocated_mw.sum(), gap_mw, atol=1e-6, rtol=0)
    assert (running.allocated_mw <= running.capacity_mw + 1e-8).all()
    return running

In [ ]:
# Use one scenario and date for the detailed country, technology and age profiles.
TARGET_YEAR = 2040
TARGET_SCENARIO = "Net Zero 2050"
detail_list = []
regional_rows = []
for fuel in ["Coal", "Gas"]:
    for region in R5_REGIONS:
        # Average the three models within this region and fuel, not across the whole world.
        target = capacity.loc[(capacity.fuel == fuel) & (capacity.region == region)
                              & (capacity.scenario == TARGET_SCENARIO) & (capacity.year == TARGET_YEAR), "capacity_gw"].mean()
        # Fill each region's own gap; spare capacity in another region cannot offset it here.
        d = stranded_units(plants[(plants.fuel == fuel) & (plants.r5 == region)], LIFETIMES, TARGET_YEAR, target)
        detail_list.append(d)
        regional_rows.append(dict(fuel=fuel, region=region, committed_gw=d.capacity_mw.sum()/1000,
                                  scenario_gw=target, allocated_gw=d.allocated_mw.sum()/1000))
# Keep all eligible units for the age chart; use only positive allocations for the profiles.
detail = pd.concat(detail_list, ignore_index=True)
early = detail[detail.allocated].copy()
regional = pd.DataFrame(regional_rows)
print(early.groupby("fuel")["allocated_mw"].sum() / 1000)
# For this case, every regional balance is positive, so regional and world gaps agree.
for fuel in ["Coal", "Gas"]:
    world_gap = gaps.loc[(gaps.fuel == fuel) & (gaps.scenario == TARGET_SCENARIO) & (gaps.year == TARGET_YEAR), "gap_gw"].iloc[0]
    assert np.isclose(regional.loc[regional.fuel == fuel, "allocated_gw"].sum(), world_gap, atol=0.001, rtol=0)


### By region

In [ ]:
# Allocated capacity (GW) by region, and each region's share of the fuel total
profile = early.pivot_table(index="r5", columns="fuel", values="allocated_mw",
                            aggfunc="sum", fill_value=0) / 1000
# Each share uses its own fuel total as the denominator.
profile["Coal share %"] = profile["Coal"] / profile["Coal"].sum() * 100
profile["Gas share %"] = profile["Gas"] / profile["Gas"].sum() * 100
profile.round(0)

In [ ]:
# Figure 3: allocated excess by region, coal and gas side by side
# Sort on the combined total so the largest regional allocation is at the top.
# This extra regional chart is shown in the notebook but not saved for the report.
region_order = (profile["Coal"] + profile["Gas"]).sort_values().index
y = np.arange(len(region_order))

fig, ax = plt.subplots(figsize=(7.2, 2.8))
ax.barh(y + 0.19, profile.loc[region_order, "Coal"], height=0.35, color=COAL_COLOR, label="Coal")
ax.barh(y - 0.19, profile.loc[region_order, "Gas"], height=0.35, color=GAS_COLOR, label="Gas")

# Write the value next to each bar
for i, region in enumerate(region_order):
    ax.text(profile.loc[region, "Coal"] + 12, i + 0.19, f"{profile.loc[region, 'Coal']:,.0f}",
            va="center", fontsize=7.5, color=GREY)
    ax.text(profile.loc[region, "Gas"] + 12, i - 0.19, f"{profile.loc[region, 'Gas']:,.0f}",
            va="center", fontsize=7.5, color=GREY)

ax.set_yticks(y)
ax.set_yticklabels([r.replace(" (R5)", "") for r in region_order])
ax.set_xlabel("Allocated excess capacity (GW)")
ax.set_title(f"{TARGET_SCENARIO}, standing in {TARGET_YEAR}", loc="left")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
ax.margins(x=0.12)


plt.show()

The regional allocations sum exactly to their own fuel-specific gaps. The world comparison is reported separately because capacity in one region cannot serve another region without further assumptions.

In [ ]:
# Show each regional balance alongside the allocation, so the totals can be checked directly.
regional.round(1)

### By country

In [ ]:
# Allocated capacity (GW) by country, the 15 largest
# Sum allocated MW, including partial boundary units, rather than counting whole units.
by_country_early = early.pivot_table(index="country", columns="fuel", values="allocated_mw",
                                     aggfunc="sum", fill_value=0) / 1000
by_country_early["Total"] = by_country_early["Coal"] + by_country_early["Gas"]
# Country shares use the combined coal-and-gas allocation across all countries.
by_country_early["share of total %"] = by_country_early["Total"] / by_country_early["Total"].sum() * 100
by_country_early = by_country_early.sort_values("Total", ascending=False)
by_country_early.head(15).round(1)

In [ ]:
# Figure 4: the 10 countries with the most allocated excess capacity
# Pick the largest ten first, then reverse the display order for horizontal bars.
top_10 = by_country_early.head(10).sort_values("Total")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.barh(top_10.index, top_10["Coal"], color=COAL_COLOR, label="Coal")
# Put gas after coal on the same bar so its full length shows the country total.
ax.barh(top_10.index, top_10["Gas"], left=top_10["Coal"], color=GAS_COLOR, label="Gas")
for i, total in enumerate(top_10["Total"]):
    ax.text(total + 10, i, f"{total:,.0f}", va="center", fontsize=7.5, color=GREY)

ax.set_xlabel("Allocated excess capacity (GW)")
ax.set_title(f"Top 10 countries, {TARGET_SCENARIO}, standing in {TARGET_YEAR}", loc="left")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
ax.margins(x=0.1)

fig.savefig(FIGURE_FOLDER / "fig4_country_profile.png")
plt.show()

### By technology

In [ ]:
# Allocated capacity by technology: capacity, share of the fuel total, and age in the target year
# Use the trackers' detailed technology labels here, rather than the broader lifetime groups.
# Unit counts include partly allocated units; capacity uses only their allocated portion.
by_tech = early.groupby(["fuel", "technology"]).agg(
    units=("unit_id", "size"),
    capacity_gw=("allocated_mw", "sum"),
    median_age=("age_in_target_year", "median"),
)
by_tech["capacity_gw"] = by_tech["capacity_gw"] / 1000
# Divide each technology by its fuel total, so the coal and gas shares each add to 100%.
by_tech["share_of_fuel_%"] = by_tech["capacity_gw"] / by_tech.groupby("fuel")["capacity_gw"].transform("sum") * 100
by_tech.sort_values(["fuel", "capacity_gw"], ascending=[True, False]).round(1)

### By age

In [ ]:
# Unit-count medians among units with positive allocated capacity.
# A fractional boundary unit counts as one; these are not capacity-weighted medians.
# Remaining life is the assumed retirement year minus 2040 for each allocated unit.
age_summary = early.groupby("fuel").agg(median_age=("age_in_target_year", "median"),
                                         median_remaining=("remaining_life_at_target", "median"))
age_summary

In [ ]:
# Figure 5: allocated and unallocated portions of target-year committed capacity.
bins = np.arange(0, 70, 2.5)
fig, axes = plt.subplots(1, 2, figsize=(7.6, 2.8))
for ax, fuel in zip(axes, ["Coal", "Gas"]):
    d = detail[detail.fuel == fuel]
    # Use the same ages twice but different MW weights to split every bin into
    # its allocated and unallocated portions, without double-counting capacity.
    ax.hist([d.age_in_target_year, d.age_in_target_year], bins=bins,
            weights=[d.allocated_mw / 1000, (d.capacity_mw - d.allocated_mw) / 1000], stacked=True,
            color=[GAS_COLOR, COAL_COLOR], label=["Allocated excess", "Not allocated"],
            edgecolor="white", linewidth=0.5)
    # The line is the unit-count median reported above; the bars themselves are weighted by GW.
    median_age = age_summary.loc[fuel, "median_age"]
    ax.axvline(median_age, color=INK, linestyle="--", linewidth=1)
    ax.text(median_age+1,ax.get_ylim()[1]*0.92,f"median {median_age:.0f} yr",fontsize=7.5)
    ax.set_title(fuel,loc="left")
    ax.set_xlabel(f"Age in {TARGET_YEAR} (years)")
    ax.grid(axis="x",visible=False)
axes[0].set_ylabel("Capacity (GW)")
axes[1].legend(fontsize=7.5)
fig.savefig(FIGURE_FOLDER / "fig5_age_profile.png")
plt.show()

## 10. Changing the lifetime assumption

We move every assumed lifetime by the same number of years (-10, -5, 0, +5, +10) and recompute the 2040 gap for each scenario.

In [ ]:
# Change lifetimes while keeping the fleet, target year and scenario capacities fixed.
rows = []
for shift in [-10, -5, 0, 5, 10]:
    # Every lifetime moved by the same number of years
    shifted_lifetimes = {}
    for group, years in LIFETIMES.items():
        shifted_lifetimes[group] = years + shift

    for fuel in ["Coal", "Gas"]:
        # Recompute committed capacity from the units; do not just scale the original gap.
        path = committed_capacity(plants[plants["fuel"] == fuel], shifted_lifetimes, [TARGET_YEAR])
        ours = path["committed_gw"].iloc[0]
        for scenario in SCENARIOS:
            theirs = scen[(scen["fuel"] == fuel) & (scen["scenario"] == scenario)
                          & (scen["year"] == TARGET_YEAR)]["capacity_gw"].iloc[0]
            rows.append({"fuel": fuel, "scenario": scenario, "lifetime_shift": shift,
                         "gap_gw": max(ours - theirs, 0)})
sens = pd.DataFrame(rows)

sens.pivot_table(index=["fuel", "scenario"], columns="lifetime_shift", values="gap_gw").round(0)

In [ ]:
# Figure 6: the 2040 gap against the lifetime shift
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
# Zero on the x-axis is the base assumption; the other points are alternative assumptions.
# These ranges are a sensitivity exercise, not statistical confidence intervals.
for ax, fuel in zip(axes, ["Coal", "Gas"]):
    for scenario, color in SCENARIO_COLORS.items():
        d = sens[(sens["fuel"] == fuel) & (sens["scenario"] == scenario)]
        ax.plot(d["lifetime_shift"], d["gap_gw"], color=color, marker="o", label=scenario)
    ax.set_title(f"{fuel}, world", loc="left")
    ax.set_xlabel("Change in assumed lifetime (years)")
    ax.set_xticks([-10, -5, 0, 5, 10])
axes[0].set_ylabel(f"Excess capacity in {TARGET_YEAR} (GW)")
axes[1].legend(fontsize=7.5)

fig.savefig(FIGURE_FOLDER / "fig6_sensitivity.png")
plt.show()

## 11. Capacity and generation

We show generation paths in EJ/year and index generation and capacity to 2025 = 100. Different paths reveal changing utilisation. They do not establish which plants close or prove that a generation-based closure estimate would be larger.

In [ ]:
# Generation from coal and gas (EJ per year), world, average of the three models
generation = ngfs[ngfs["variable"].str.startswith("Secondary Energy")
                  & (ngfs["region"] == "World")
                  & ngfs["scenario"].isin(SCENARIOS)
                  & (ngfs["year"] >= 2025) & (ngfs["year"] <= 2050)]
generation = generation.groupby(["fuel", "scenario", "year"], as_index=False)["value"].mean()

# Index both generation and capacity to 2025 = 100
# Indexing lets us compare changes in EJ/year and GW on the same scale.
comparison = generation.merge(scen, on=["fuel", "scenario", "year"])
comparison["generation_index"] = 0.0
comparison["capacity_index"] = 0.0
for fuel in ["Coal", "Gas"]:
    for scenario in SCENARIOS:
        rows = (comparison["fuel"] == fuel) & (comparison["scenario"] == scenario)
        # Use each fuel-scenario pair's own 2025 value as its baseline of 100.
        start = comparison[rows & (comparison["year"] == 2025)].iloc[0]
        comparison.loc[rows, "generation_index"] = comparison.loc[rows, "value"] / start["value"] * 100
        comparison.loc[rows, "capacity_index"] = comparison.loc[rows, "capacity_gw"] / start["capacity_gw"] * 100

# Show the indices at three dates, then the absolute generation levels underneath.
# Different paths can reflect utilisation changes; they do not identify particular closures.
result = comparison[comparison["year"].isin([2030, 2040, 2050])]
display(result.set_index(["fuel", "scenario", "year"])[["generation_index", "capacity_index"]].round(0))
# Absolute generation paths, as requested in the track brief.
display(generation.pivot_table(index=["fuel", "scenario"], columns="year", values="value").round(2))

In [ ]:
# Generate report tables and key values from this run, avoiding manual transcription.
def tex_number(value, decimals=0):
    # Braces stop LaTeX from adding extra space after the thousands-separator comma.
    return f"{value:,.{decimals}f}".replace(",", "{,}")

def write_table(filename, frame, decimals=1):
    # Escape text labels for LaTeX and apply the same decimal format to all numeric columns.
    text = frame.to_latex(index=False, escape=True, float_format=lambda x: f"{x:,.{decimals}f}")
    (FIGURE_FOLDER / filename).write_text(text, encoding="utf-8")

# Export the retirement-age summary alongside the assumed lifetimes.
life_table = table2.reset_index()[["tech_group", "closed_units", "median", "p25", "p75", "assumed"]]
life_table.columns = ["Technology", "Closures", "Median", "25th pct.", "75th pct.", "Assumed"]
write_table("lifetimes.tex",life_table)

# The report pools technologies into the lifetime groups; the detailed breakdown is above.
tech_table = early.groupby(["fuel","tech_group"]).allocated_mw.sum().div(1000).reset_index(name="Allocated GW")
tech_table["Share within fuel (%)"] = tech_table["Allocated GW"] / tech_table.groupby("fuel")["Allocated GW"].transform("sum") * 100
tech_table.columns = ["Fuel","Technology","Allocated GW","Share within fuel (%)"]
write_table("technology.tex",tech_table)

# Keep models separate here to report their range at the base lifetime assumptions.
model_gaps = world[(world.year == TARGET_YEAR) & (world.scenario == TARGET_SCENARIO)].copy()
model_gaps = model_gaps.merge(committed[committed.year == TARGET_YEAR][["fuel","committed_gw"]],on="fuel")
model_gaps["gap_gw"] = (model_gaps.committed_gw - model_gaps.capacity_gw).clip(lower=0)
display(model_gaps[["model","fuel","gap_gw"]].round(1))

# Store each report number with its display precision. The calculations stay unrounded.
values = {}
for fuel in ["Coal","Gas"]:
    values[fuel+"Age"] = (age_summary.loc[fuel,"median_age"],0)
    values[fuel+"Remaining"] = (age_summary.loc[fuel,"median_remaining"],0)
    values[fuel+"ModelMin"] = (model_gaps.loc[model_gaps.fuel == fuel,"gap_gw"].min(),0)
    values[fuel+"ModelMax"] = (model_gaps.loc[model_gaps.fuel == fuel,"gap_gw"].max(),0)
    values[fuel+"Eligible"] = (committed.loc[(committed.fuel == fuel)&(committed.year == BASE_YEAR),"committed_gw"].iloc[0],0)
for name,country in [("China","China"),("India","India"),("US","United States")]:
    values[name+"Allocation"] = (by_country_early.loc[country,"Total"],0)
values["ChinaShare"] = (by_country_early.loc["China","share of total %"],1)
values["AsiaCoalShare"] = (profile.loc["Asia (R5)","Coal share %"],1)
values["MEAGasShare"] = (profile.loc["Middle East & Africa (R5)","Gas share %"],1)
values["OECDGasShare"] = (profile.loc["OECD & EU (R5)","Gas share %"],1)
values["ConstructionCoal"] = (construction_coal_gw,0)
  
# Write named LaTeX commands so the report can insert these values without retyping them.
lines = ["\\newcommand{\\"+name+"}{"+tex_number(value,decimals)+"}" for name,(value,decimals) in values.items()]
(FIGURE_FOLDER / "report_values.tex").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Python:",sys.version.split()[0],"| pandas:",pd.__version__,"| NumPy:",np.__version__)


## 12. Econometrics: observed retirement ages

We estimate whether recorded retirement ages differ by technology and unit size. This is a separate descriptive exercise: it does not change the assumed lifetimes or any of the capacity-gap calculations above.

The dependent variable is retirement age in years. Model 1 uses technology indicators, with coal as the reference. Model 2 adds log-base-2 capacity in MW, R5-region indicators and retirement-year indicators. A size coefficient therefore measures the difference in retirement age associated with doubling capacity, holding the included characteristics fixed.

We use the common 2020–2026 retirement window for both fuels and require positive, known capacity. This makes the dates comparable, but the sample still contains only recorded closures. It does not identify the eventual technical lifetimes of operating plants or a causal effect of technology.


In [ ]:
# Use the same years for coal and gas; the gas tracker has no earlier closure record.
import statsmodels.formula.api as smf

econ_sample = retired.loc[
    retired.retired_year.between(2020, BASE_YEAR)
    & retired.capacity_mw.notna() & (retired.capacity_mw > 0)
].copy()

# A one-unit change in log2(MW) means a doubling in physical capacity.
econ_sample["log2_capacity"] = np.log2(econ_sample.capacity_mw)
econ_sample["closure_year"] = econ_sample.retired_year.astype(int)
assert econ_sample[["lifetime", "log2_capacity", "tech_group", "r5", "country"]].notna().all().all()

# Show the sample behind the regression, including the small residual technology group.
print(f"Regression sample: {len(econ_sample):,} closures in {econ_sample.country.nunique()} countries or areas")
display(econ_sample.groupby("tech_group").agg(
    closures=("unit_id", "size"), mean_age=("lifetime", "mean"),
    mean_capacity_mw=("capacity_mw", "mean"),
).round(1))


In [ ]:
# Model 1 describes technology differences in average retirement age relative to coal.
technology_term = 'C(tech_group, Treatment(reference="Coal (all)"))'
econ_formula_1 = "lifetime ~ " + technology_term

# Model 2 asks whether those differences remain after accounting for size, region and year.
# Region and year indicators allow each category its own intercept adjustment.
econ_formula_2 = econ_formula_1 + " + log2_capacity + C(r5) + C(closure_year)"

# HC3 standard errors allow the residual variance to differ across units.
# The coefficients are ordinary least squares estimates; only their uncertainty calculation changes.
econ_models = {
    "Technology only": smf.ols(econ_formula_1, data=econ_sample).fit(cov_type="HC3", use_t=True),
    "With controls": smf.ols(econ_formula_2, data=econ_sample).fit(cov_type="HC3", use_t=True),
}
for name, model in econ_models.items():
    # Full column rank and positive residual degrees of freedom are needed for these comparisons.
    assert np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1]
    assert model.df_resid > 0 and np.isfinite(model.bse).all()
    print(f"{name}: N = {int(model.nobs):,}, R-squared = {model.rsquared:.3f}")

# Put coefficients, standard errors and confidence intervals in one readable table.
econ_rows = []
for name, model in econ_models.items():
    interval = model.conf_int()
    for term in model.params.index:
        econ_rows.append(dict(model=name, term=term, coefficient=model.params[term],
                              standard_error=model.bse[term], lower_95=interval.loc[term, 0],
                              upper_95=interval.loc[term, 1], p_value=model.pvalues[term]))
econ_results = pd.DataFrame(econ_rows)
display(econ_results.round(3))


In [ ]:
# Units in the same country can share policies and market conditions.
# Check how uncertainty changes if errors are allowed to be correlated within countries.
# This changes standard errors, not the OLS coefficients or the fitted sample.
econ_country_clusters = econ_sample.country.nunique()
assert econ_country_clusters > 1
econ_clustered = smf.ols(econ_formula_2, data=econ_sample).fit(
    cov_type="cluster", cov_kwds={"groups": econ_sample.country, "use_correction": True}, use_t=True
)
econ_adjusted = econ_models["With controls"]
assert np.allclose(econ_clustered.params, econ_adjusted.params)
cluster_interval = econ_clustered.conf_int()
econ_cluster_check = pd.DataFrame({
    "coefficient": econ_adjusted.params,
    "HC3 SE": econ_adjusted.bse,
    "country-clustered SE": econ_clustered.bse,
    "cluster lower 95": cluster_interval[0],
    "cluster upper 95": cluster_interval[1],
})
print(f"Country-clustered check: {econ_country_clusters} clusters")
display(econ_cluster_check.round(3))


In [ ]:
# Export only the compact table used in the paper. Full results remain displayed above.
econ_terms = []
for group in ["Gas: combined cycle", "Gas: gas turbine", "Gas: steam turbine", "Gas: other/unknown"]:
    term = next(t for t in econ_adjusted.params.index if t.endswith("[T." + group + "]"))
    econ_terms.append((group.replace("Gas: ", "Gas, ") + " (vs coal)", term))
econ_terms.append(("Capacity doubling", "log2_capacity"))

# Entries are coefficient differences in years, not predicted retirement ages.
# Negative technology coefficients mean younger retirement than coal.
# Each coefficient is followed by its HC3 standard error in parentheses.
# The intercept and control coefficients remain visible in the full results above.
row_end = chr(92) * 2
latex_slash = chr(92)
econ_lines = [latex_slash + "begin{tabular}{lrr}", latex_slash + "toprule",
              "Coefficient & Technology only & With controls " + row_end, latex_slash + "midrule"]
for label, term in econ_terms:
    coefficients = []
    for model in econ_models.values():
        coefficients.append(f"{model.params[term]:.2f} ({model.bse[term]:.2f})" if term in model.params else "--")
    econ_lines.append(label + " & " + " & ".join(coefficients) + " " + row_end)
econ_lines.extend([latex_slash + "midrule", "Region and year indicators & No & Yes " + row_end,
                   "Closures & " + " & ".join(str(int(m.nobs)) for m in econ_models.values()) + " " + row_end,
                   "R-squared & " + " & ".join(f"{m.rsquared:.3f}" for m in econ_models.values()) + " " + row_end,
                   latex_slash + "bottomrule", latex_slash + "end{tabular}"])
(FIGURE_FOLDER / "econometrics.tex").write_text("\n".join(econ_lines) + "\n", encoding="utf-8")

# Display the key effects and both confidence intervals for the interpretation in the report.
econ_key_terms = [term for _, term in econ_terms if term != "Intercept"]
display(econ_cluster_check.loc[econ_key_terms].round(3))
print("statsmodels version:", __import__("statsmodels").__version__)
